# 🧪 UCB-GLOBES MS Data Visualization and Comparison Tool


Welcome to the **UCB-GLOBES Mass Spectrometry Data Visualization and Comparison Tool**! This notebook is designed to help you **interactively explore and compare** mass spectra with ease.

---

### 🔍 What You Can Do with This Tool:

- **Input two mass spectra** and **visually compare** them side-by-side  
- **Generate a cosine similarity matrix** from **multiple spectra** of your choice  
- Gain insights into **spectral similarity and patterns**

---

### 📘 How to Use This Notebook

The following cells will guide you **step-by-step** through the process, including:

1. Uploading or selecting mass spectra  
2. Plotting spectra for visual inspection  
3. Calculating cosine similarity  
4. Interpreting results from the similarity matrix

---

Feel free to begin at the next cell and follow along — all inputs and actions are clearly explained. Happy analyzing! 🔬📊

Original notebook author Stephanie Xu 7/3/2025, Last modified by Lindsay D. Yee 11/19/2025 


In [ ]:
#Action: Run the cell

#Purpose : Install all the libraries needed for later use
!pip install requests pandas gdown numpy
!pip install ipywidgets
!pip install --upgrade ipywidgets
!pip install plotly



In [ ]:
#Action: Run the cell

#Purpose : Import all the libraries needed for later use

import re
import requests
import pandas as pd
from io import StringIO
import gdown
import plotly.graph_objects as go
import ipywidgets as widgets
from IPython.display import display, clear_output
from math import sqrt
import plotly.io as pio
from datetime import datetime
from IPython.display import HTML
import io
import numpy as np
import plotly.offline as pyo


In [ ]:
#Action: Run the cell / replace the file_id with new input if necessary
pio.renderers.default = 'notebook'

file_id = '1gvc36AaicXx_9AzFZE3eRfFoG7fIamEl'
url = f'https://drive.google.com/uc?id={file_id}'
output = 'database.csv'
gdown.download(url, output, quiet=False)

print(f"✅ Data last pulled on data: {datetime.now().strftime('%Y-%m-%d')}")

database = pd.read_csv(output, low_memory=False, on_bad_lines='skip')
database.head()



In [ ]:
ROWS_PER_PAGE = 10



column_dropdown = widgets.Dropdown(
    options=database.columns.tolist(),
    description="Filter by:",
    layout=widgets.Layout(width='50%')
)

filter_text = widgets.Text(
    description="Value:",
    placeholder="Enter value to match (partial OK)",
    layout=widgets.Layout(width='50%')
)

filter_btn = widgets.Button(
    description="Filter Data",
    button_style='info'
)

clear_filter_btn = widgets.Button(
    description="Clear Filter",
    button_style='warning'
)

prev_btn = widgets.Button(description="Previous", button_style='')
next_btn = widgets.Button(description="Next", button_style='')

filter_output = widgets.Output()
pagination_output = widgets.Output()

filtered_df = pd.DataFrame()
current_page = 0

exclude_cols = ["Publications", "MS", "X-Values", "Y-Values"]

def display_page():
    global filtered_df, current_page
    with filter_output:
        clear_output()
        start = current_page * ROWS_PER_PAGE
        end = start + ROWS_PER_PAGE
        if not filtered_df.empty:
            print(f"✅ Showing rows {start + 1} to {min(end, len(filtered_df))} of {len(filtered_df)}:")
            display_df = filtered_df.drop(columns=[c for c in exclude_cols if c in filtered_df.columns])
            html_table = display_df.iloc[start:end].to_html()
            scrollable_html = f'<div style="height:400px; overflow-y:auto; border:1px solid gray;">{html_table}</div>'
            display(HTML(scrollable_html))
        else:
            print("⚠️ No matching rows.")

    with pagination_output:
        clear_output()
        if len(filtered_df) > ROWS_PER_PAGE:
            buttons = []
            if current_page > 0:
                buttons.append(prev_btn)
            if (current_page + 1) * ROWS_PER_PAGE < len(filtered_df):
                buttons.append(next_btn)
            display(widgets.HBox(buttons))


def on_filter_clicked(_):
    global filtered_df, current_page
    with filter_output:
        clear_output()
        col = column_dropdown.value
        val = filter_text.value.strip().lower()
        if not val:
            print("❗ Please enter a filter value.")
            return
        filtered_df = database[database[col].astype(str).str.lower().str.contains(val, na=False)]
        current_page = 0
    display_page()


def on_clear_filter_clicked(_):
    global filtered_df, current_page
    with filter_output:
        clear_output()
        print("🔄 Filter cleared.")
    with pagination_output:
        clear_output()
    filtered_df = pd.DataFrame()
    current_page = 0

def on_next_clicked(_):
    global current_page
    current_page += 1
    display_page()

def on_prev_clicked(_):
    global current_page
    if current_page > 0:
        current_page -= 1
    display_page()


filter_btn.on_click(on_filter_clicked)
clear_filter_btn.on_click(on_clear_filter_clicked)
next_btn.on_click(on_next_clicked)
prev_btn.on_click(on_prev_clicked)


display(widgets.VBox([
    widgets.HBox([column_dropdown, filter_text]),
    widgets.HBox([filter_btn, clear_filter_btn]),
    filter_output,
    pagination_output
]))

In [ ]:
# Action: Run the cell and follow the instruction at the bottom

pio.renderers.default = 'iframe'
pyo.init_notebook_mode(connected=True)

def cosine_similarity(y1, y2):
    dot = sum(a * b for a, b in zip(y1, y2))
    norm1 = sqrt(sum(a ** 2 for a in y1))
    norm2 = sqrt(sum(b ** 2 for b in y2))
    return dot / (norm1 * norm2) if norm1 and norm2 else 0

def align_mass_spectra(x1, y1, x2, y2):
    all_x = sorted(set(x1 + x2))
    y1_aligned = [y1[x1.index(x)] if x in x1 else 0 for x in all_x]
    y2_aligned = [y2[x2.index(x)] if x in x2 else 0 for x in all_x]
    return all_x, y1_aligned, y2_aligned

def plot_mass_spectrum(x, y, name):
    fig = go.Figure()
    fig.add_trace(go.Bar(x=x, y=y, name=name))
    fig.update_layout(title=f"Mass Spectrum for {name}",
                      xaxis_title='m/z',
                      yaxis_title='Relative Abundance')
    fig.show()

def plot_mirrored_spectrum(x1, y1, name1, x2, y2, name2):
    y2_neg = [-v for v in y2]
    fig = go.Figure()

    fig.add_trace(go.Bar(x=x1, y=y1, name=name1, marker_color='blue'))
    fig.add_trace(go.Bar(x=x2, y=y2_neg, name=name2, marker_color='red'))

    fig.update_layout(title=f"Mirrored Plot: {name1} vs {name2}",
                      xaxis_title='m/z',
                      yaxis_title='Relative Abundance',
                      barmode='overlay',
                      bargap=0.1)
    fig.show()

def plot_subtraction(x, y_diff, name1, name2):
    colors = ['blue' if val > 0 else 'red' for val in y_diff]

    fig = go.Figure()
    fig.add_trace(go.Bar(x=x, y=y_diff, marker_color=colors,
                         name=f"{name1} - {name2}"))

    fig.update_layout(title=f"Subtraction Plot: {name1} - {name2}",
                      xaxis_title='m/z',
                      yaxis_title='Difference in Relative Abundance')
    fig.show()

def plot_scatter(y1, y2, name1, name2):
    similarity = cosine_similarity(y1, y2)
    
    min_val = min(min(y1), min(y2))
    max_val = max(max(y1), max(y2))
    
    fig = go.Figure()
    fig.add_trace(go.Scatter(x=y1, y=y2, mode='markers', name='Points'))
    fig.add_trace(go.Scatter(x=[min_val, max_val],y=[min_val, max_val],mode='lines',line=dict(dash='dash', color='red'),name='1:1 slope line'))
    
    fig.update_layout(
        title=f"Scatter Plot: {name1} vs {name2}<br>Cosine Similarity = {similarity:.3f}",
        xaxis_title=f"Intensity: {name1}",
        yaxis_title=f"Intensity: {name2}",
        legend=dict(x=0.01, y=0.99)
    )
    
    fig.show()


def parse_array_string(arr_str):
    clean_str = arr_str.strip('[]').strip()
    parts = re.split(r'\s+', clean_str)
    return [float(p) for p in parts if p]

def get_spectrum_data(identifier):
    filtered = database[database['UID'].astype(str) == identifier.strip()]
    if filtered.empty:
        filtered = database[database['Name'].str.lower() == identifier.lower()]
    if filtered.empty:
        return None, None, None
    row = filtered.iloc[0]
    x = parse_array_string(row['X-Values'])
    y = parse_array_string(row['Y-Values'])
    return row['Name'], x, y

output_area = widgets.Output()

def on_compare_clicked(_):
    with output_area:
        clear_output(wait=True)
        name1 = input1.value.strip()
        name2 = input2.value.strip()
        data1 = get_spectrum_data(name1)
        data2 = get_spectrum_data(name2)

        if data1[0] is None or data2[0] is None:
            print("Error: One or both spectrum identifiers not found. Check Name or UID.")
            return

        name1, x1, y1 = data1
        name2, x2, y2 = data2

        plot_mass_spectrum(x1, y1, name1)
        plot_mass_spectrum(x2, y2, name2)

        x_aligned, y1_aligned, y2_aligned = align_mass_spectra(x1, y1, x2, y2)

        plot_mirrored_spectrum(x1, y1, name1, x2, y2, name2)

        y_diff = [a - b for a, b in zip(y1_aligned, y2_aligned)]
        plot_subtraction(x_aligned, y_diff, name1, name2)

        plot_scatter(y1_aligned, y2_aligned, name1, name2)

def on_clear_clicked(_):
    with output_area:
        clear_output(wait=True)

input1 = widgets.Text(description="Spectrum 1:", placeholder="e.g. levoglucosan, 3TMS")
input2 = widgets.Text(description="Spectrum 2:", placeholder="e.g. LDY-1")

compare_btn = widgets.Button(description="Compare Spectra", button_style='success')
clear_btn = widgets.Button(description="Clear Output", button_style='danger')

compare_btn.on_click(on_compare_clicked)
clear_btn.on_click(on_clear_clicked)

display(widgets.VBox([
    widgets.HTML("<b>Enter the spectrum identifiers (Name or UID)"),
    input1,
    input2,
    compare_btn,
    clear_btn,
    output_area
]))


In [ ]:

def cosine_similarity(y1, y2):
    dot = sum(a * b for a, b in zip(y1, y2))
    norm1 = sqrt(sum(a ** 2 for a in y1))
    norm2 = sqrt(sum(b ** 2 for b in y2))
    return dot / (norm1 * norm2) if norm1 and norm2 else 0

def get_retention_index(name):
    row = database[database['Name'] == name]
    if row.empty:
        return None
    value = row.iloc[0]['Retention_index']
    if isinstance(value, str) and '=' in value:
        return float(value.split('=')[-1])



def parse_array_string(arr_str):
    clean_str = arr_str.strip('[]').strip()
    parts = re.split(r'\s+', clean_str)
    return [float(p) for p in parts if p]

def get_spectrum_by_name(name):
    filtered = database[database['Name'] == name]
    if filtered.empty:
        return None
    row = filtered.iloc[0]
    y = parse_array_string(row['Y-Values'])
    return y

def calculate_cosine_similarity_matrix(names):
    spectra_data = []
    retention_indices = []

    for n in names:
        row = database[database['Name'] == n]
        if row.empty:
            raise ValueError(f"Spectrum '{n}' not found or invalid.")
        x = parse_array_string(row.iloc[0]['X-Values'])
        y = parse_array_string(row.iloc[0]['Y-Values'])
        ri = get_retention_index(n)
        spectra_data.append((n, x, y))
        retention_indices.append(ri)

    size = len(spectra_data)
    similarity_matrix = np.zeros((size, size))
    ri_diff_matrix = np.zeros((size, size))

    for i in range(size):
        for j in range(size):
            name1, x1, y1 = spectra_data[i]
            name2, x2, y2 = spectra_data[j]
            _, y1_aligned, y2_aligned = align_mass_spectra(x1, y1, x2, y2)
            similarity_matrix[i, j] = cosine_similarity(y1_aligned, y2_aligned)
            if retention_indices[i] is not None and retention_indices[j] is not None:
                ri_diff_matrix[i, j] = abs(retention_indices[i] - retention_indices[j])
            else:
                ri_diff_matrix[i, j] = float('nan')
                
    return similarity_matrix, ri_diff_matrix




all_names = sorted(database['Name'].unique())

search_box = widgets.Text(placeholder='Enter Name value to search (partial OK)', description='Search:', layout=widgets.Layout(width='50%'))
search_results = widgets.SelectMultiple(options=[], description='Results:', rows=6, layout=widgets.Layout(width='50%'))
selected_rows = widgets.SelectMultiple(options=[], description='Selected:', rows=6, layout=widgets.Layout(width='50%'))
add_btn = widgets.Button(description='Add →', button_style='info')
remove_btn = widgets.Button(description='← Remove', button_style='warning')
threshold_slider = widgets.FloatSlider(value=0.7, min=0, max=1, step=0.01, description='Threshold:', layout=widgets.Layout(width='50%'))
calculate_btn = widgets.Button(description='Calculate Similarity', button_style='success')
output = widgets.Output(layout={'border': '1px solid gray', 'height': '400px', 'overflow_y': 'auto'})



def update_search_results(change):
    query = change['new'].lower().strip()
    if not query:
        search_results.options = []
        return
    matches = [name for name in all_names if query in name.lower()]
    search_results.options = matches[:100]

def add_selected(_):
    current = list(selected_rows.options)
    to_add = [name for name in search_results.value if name not in current]
    selected_rows.options = current + to_add

def remove_selected(_):
    current = list(selected_rows.options)
    to_remove = set(selected_rows.value)
    selected_rows.options = [name for name in current if name not in to_remove]

def render_similarity_matrix(names, sim_matrix, ri_matrix, threshold):
    with output:
        clear_output()
        size = len(names)
        html = '<table style="border-collapse: collapse; text-align: center;">'
        html += '<tr><th></th>'
        for n in names:
            html += f'<th style="border:1px solid black; padding:4px;">{n}</th>'
        html += '</tr>'
        for i in range(size):
            html += f'<tr><th style="border:1px solid black; padding:4px;">{names[i]}</th>'
            for j in range(size):
                sim_val = sim_matrix[i, j]
                ri_diff = ri_matrix[i, j]
                bg = 'rgba(128,128,128,0.5)' if sim_val >= threshold else 'rgba(255,255,255,0.5)'
                if np.isnan(ri_diff):
                    cell_text = f"{sim_val:.3f} | ΔRI=N/A"
                else:
                    cell_text = f"{sim_val:.3f} | ΔRI={ri_diff:.1f}"
                html += f'<td style="border:1px solid black; padding:4px; background-color:{bg};">{cell_text}</td>'
            html += '</tr>'
        html += '</table>'
        display(widgets.HTML(value=html))


def calculate_clicked(_):
    selected_set = set(selected_rows.options)

    filtered_names = set()
    if use_filtered_checkbox.value:
        if 'filtered_df' in globals() and not filtered_df.empty:
            filtered_names = set(filtered_df['Name'].tolist())

    combined_names = selected_set.union(filtered_names)

    if len(combined_names) < 2:
        with output:
            clear_output()
            print("⚠️ Need at least two spectra to calculate similarity.")
        return

    names = list(combined_names)
    sim_matrix, ri_matrix = calculate_cosine_similarity_matrix(names)
    render_similarity_matrix(names, sim_matrix, ri_matrix, threshold_slider.value)

   




            
    
use_filtered_checkbox = widgets.Checkbox(
    value=False,
    description="Add in all filtered rows above",
    indent=False,
    layout=widgets.Layout(width='200px')
)


search_box.observe(update_search_results, names='value')
add_btn.on_click(add_selected)
remove_btn.on_click(remove_selected)
calculate_btn.on_click(calculate_clicked)

button_box = widgets.VBox([add_btn, remove_btn])

display(search_box)
display(widgets.HBox([search_results, button_box, selected_rows]))
threshold_note = widgets.HTML(
    value="""
    <div style="font-size: 90%; color: gray; padding-left: 15px;">
        <b>Note:</b> Values <b>≥ threshold</b> are shaded gray to highlight strong similarity.
    </div>
    """
)

display(use_filtered_checkbox)
display(widgets.HBox([threshold_slider, threshold_note]))
display(widgets.HTML(
    value="<i>Each cell shows: cosine similarity | ΔRI (absolute retention index difference)</i>",
    layout=widgets.Layout(padding="5px 0px 0px 15px")
))
display(calculate_btn)
display(output)



### Confirm Matches

In [ ]:
display(widgets.HTML(
    value="""
    <b>Instructions:</b><br>
    Enter two UID values (e.g., <code>LDY-1234</code>) and choose one of the options:<br>
    <ul>
        <li><b>Confirmed Match</b> → Status = 1</li>
        <li><b>Unconfirmed</b> → Status = 0</li>
        <li><b>Confirmed Not a Match</b> → Status = -1</li>
    </ul>
    Click the download button to save the entries as a CSV file.
    """
))

uid1_input = widgets.Text(description="UID 1:", placeholder="e.g. LDY-1234")
uid2_input = widgets.Text(description="UID 2:", placeholder="e.g. LDY-5678")

confirm_btn = widgets.Button(description="Confirmed Match", button_style='success')
unconfirmed_btn = widgets.Button(description="Unconfirmed", button_style='warning')
not_match_btn = widgets.Button(description="Confirmed Not Match", button_style='danger')

download_btn = widgets.Button(description="Download CSV", button_style='primary')
match_output = widgets.Output()

match_table = pd.DataFrame(columns=["MS1", "MS2", "Confirm Match Status"])
match_table_display = widgets.Output()

def add_entry(status):
    global match_table
    uid1 = uid1_input.value.strip()
    uid2 = uid2_input.value.strip()
    
    if not (uid1.startswith("LDY-") and uid2.startswith("LDY-")):
        with match_output:
            clear_output()
            print("⚠️ UIDs must be in the format 'LDY-####'")
        return
    
    new_row = {"MS1": uid1, "MS2": uid2, "Confirm Match Status": status}
    match_table = pd.concat([match_table, pd.DataFrame([new_row])], ignore_index=True)
    
    with match_output:
        clear_output()
        print(f"✅ Added: {uid1} - {uid2} (Status: {status})")
    
    with match_table_display:
        clear_output()
        display(match_table)

def on_confirm(_):
    add_entry(1)

def on_unconfirmed(_):
    add_entry(0)

def on_not_match(_):
    add_entry(-1)

def on_download_csv(_):
    filename = f"spectral_matches.csv"
    match_table.to_csv(filename, index=False)
    with match_output:
        clear_output()
        print(f"✅ File saved as: {filename}, see file in your directory")

confirm_btn.on_click(on_confirm)
unconfirmed_btn.on_click(on_unconfirmed)
not_match_btn.on_click(on_not_match)
download_btn.on_click(on_download_csv)

display(widgets.HBox([uid1_input, uid2_input]))
display(widgets.HBox([confirm_btn, unconfirmed_btn, not_match_btn]))
display(download_btn)
display(match_output)
display(match_table_display)


## 📚 References

Harvey, D. J., & Vouros, P. (2020). *Mass Spectrometric Fragmentation of Trimethylsilyl and Related Alkylsilyl Derivatives*. _Mass Spectrometry Reviews, 39_(1–2), 105–211. [https://doi.org/10.1002/mas.21590](https://doi.org/10.1002/mas.21590)  

Jeon, S., Walker, M. J., Sueper, D. T., Day, D. A., Handschy, A. V., Jimenez, J. L., & Williams, B. J. (2023). *A searchable database and mass spectral comparison tool for the AMS and ACSM*. _Atmospheric Measurement Techniques, 16_(24), 6075–6095. [https://doi.org/10.5194/amt-16-6075-2023](https://doi.org/10.5194/amt-16-6075-2023)  

Mallard, W. G., Andriamaharavo, N. R., Mirokhin, Y. A., Halket, J. M., & Stein, S. E. (2014). *Creation of libraries of recurring mass spectra from large data sets assisted by a dual-column workflow*. _Analytical Chemistry, 86_(20), 10231–10238. [https://doi.org/10.1021/ac502379x](https://doi.org/10.1021/ac502379x)  

McGlynn, D. F., Yee, L. D., Garraffo, H. M., Geer, L. Y., Mak, T. D., Mirokhin, Y. A., Tchekhovskoi, D. V., Jen, C. N., Goldstein, A. H., Kearsley, A. J., & Stein, S. E. *New Library-Based Methods for Nontargeted Compound Identification by GC-EI-MS*. *(Publication details incomplete)*  

Moorthy, A. S., Wallace, W. E., Kearsley, A. J., Tchekhovskoi, D. V., & Stein, S. E. (2017). *Combining Fragment-Ion and Neutral-Loss Matching during Mass Spectral Library Searching*. _Analytical Chemistry, 89_(24), 13261–13268. [https://doi.org/10.1021/acs.analchem.7b03320](https://doi.org/10.1021/acs.analchem.7b03320)  

NIST Mass Spec Data Center, S.E. Stein (Director). (2012). *Mass Spectra*. In: Linstrom, P. & Mallard, W. M. (Eds.), _NIST Chemistry WebBook, NIST Standard Reference Database Number 69_. National Institute of Standards and Technology. [Available Online](https://webbook.nist.gov/chemistry/)  

Stein, S. E. (1994). *Estimating probabilities of correct identification from results of mass spectral library searches*. _Journal of the American Society for Mass Spectrometry, 5_(4), 316–323. [https://doi.org/10.1016/1044-0305(94)85022-4](https://doi.org/10.1016/1044-0305(94)85022-4)  

Stein, S. E. (2012). *Mass Spectral Reference Libraries: An Ever-Expanding Resource for Chemical Identification*. _Analytical Chemistry, 84_, 7274–7282. [https://doi.org/10.1021/ac301205z](https://doi.org/10.1021/ac301205z)  

Stein, S. E., & Heller, D. N. (2006). *On the Risk of False Positive Identification Using Multiple Ion Monitoring in Qualitative Mass Spectrometry*. _Journal of the American Society for Mass Spectrometry, 17_(6), 823–835. [https://doi.org/10.1016/j.jasms.2006.02.021](https://doi.org/10.1016/j.jasms.2006.02.021)  

Stein, S. E., & Scott, D. R. (1994). *Optimization and testing of mass spectral library search algorithms for compound identification*. _Journal of the American Society for Mass Spectrometry, 5_(9), 859–866. [https://doi.org/10.1016/1044-0305(94)87009-8](https://doi.org/10.1016/1044-0305(94)87009-8)  

Wallace, W. E., Ji, W., Tchekhovskoi, D. V., Phinney, K. W., & Stein, S. E. (2017). *Mass Spectral Library Quality Assurance by Inter-Library Comparison*. _Journal of the American Society for Mass Spectrometry, 28_(4), 733–738. [https://doi.org/10.1007/s13361-016-1589-4](https://doi.org/10.1007/s13361-016-1589-4)


## 📬 Contacts

For questions, comments, or collaboration inquiries, please reach out to:

---

**👩‍🔬 Lindsay Yee**  
*Project Scientist, Exceptional PI Status*   
*Allen Goldstein Research Group*

Email: lindsay.yee@berkeley.edu

---